# Lab 4 — DATS 6103

**Name:**

This notebook mirrors the lab page. Everything here also runs in the browser at
<https://www.smajhi.com/DATS-6103/labs/numpy-II.html>, where the self-checks and the timers live; work in whichever you
prefer. The discussion half of the lab is not in this file — it happens away
from the keyboard.

**Conditions.** No AI assistants. The official Python, NumPy, Pandas,
Matplotlib and scikit-learn documentation and the course notes may be consulted
freely.

Run the self-check under each answer. It tells you whether your answer is right
without telling you what the answer is.

## Part A—Discussion

**Not submitted, and not written at the keyboard.** Work these in pairs, out loud, one of you holding the rubric card. The rubric cards are on the lab page: <https://www.smajhi.com/DATS-6103/labs/numpy-II.html>.

### A1. Why not just invert the matrix?

*Suggested: 5 minutes.*

*Phone screen, quantitative / ML role.*

> "I am reading your code and I see
>
> ```
> x = np.linalg.inv(A) @ b
> ```
>
> What would you change, and why? And is there ever a case where computing the inverse is the right call?"

### A2. How many of those digits do you believe?

*Suggested: 5 minutes.*

*Onsite, quantitative / ML role.*

> "You solve $A\mathbf{x}=\mathbf{b}$ and print the answer to every digit NumPy shows you. There is no warning of any kind. Someone tells you $\kappa(A)\approx 10^{10}$. How many of those digits do you believe, and how did you decide?"

### A3. Imaginary parts on a covariance matrix

*Suggested: 5 minutes.*

*Onsite, ML-leaning role.*

> "You call `np.linalg.eig` on a covariance matrix and the eigenvalues come back complex—imaginary parts around $10^{-17}$. What happened, and what do you do about it?"

### A4. Two implementations of PCA that disagree

*Suggested: 5 minutes.*

*Take-home follow-up discussion, data scientist.*

> "Two teammates implement PCA on the same data. Both center it first. One takes `eigh` of its covariance matrix, $B^\top B/(m-1)$; the other takes `svd` of the centered matrix $B$. The top two components agree to eight digits. The fourth does not agree at all. Which do you trust, and what do you write in the code review?"

### A5. Explain the SVD, then justify the centering

*Suggested: 5 minutes.*

*Onsite, ML role.*

> "Explain the SVD to me as if I am a strong engineer who has never used it—what it gives you, and what you would use it for. Then tell me why PCA starts by subtracting the column means."

## Part B—Build

### B1. Solve it, and measure both ways of being wrong

*Suggested: 8 minutes.*

The setup cell builds $H$, the $8\times8$ **Hilbert matrix**. Its entry in row $i$, column $j$ is $1/(i+j-1)$, and it is a standard example of a matrix that is hard to solve with accurately. It also builds a right-hand side you know the answer to in advance: $\mathbf{b} = H\mathbf{1}$, where $\mathbf{1}$ is the vector of eight ones. So the exact solution of $H\mathbf{x} = \mathbf{b}$ is $\mathbf{x} = \mathbf{1}$, stored as `x_true`, and you can measure how far each method lands from it. Compute:

- `kappa`—the condition number of $H$
- `x_solve`—the solution from `np.linalg.solve`
- `x_inv`—the solution from `inv(H) @ b`
- `res_solve`, `res_inv`—the **residuals** $\|H\mathbf{x}-\mathbf{b}\|_2$ for each
- `err_solve`, `err_inv`—the **relative forward errors** $\|\mathbf{x}-\mathbf{1}\|_2 / \|\mathbf{1}\|_2$ for each

Residual and error are different questions. Look at all four numbers before you go on.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
def hilbert(n):
    """The n-by-n Hilbert matrix, H[i, j] = 1 / (i + j + 1)."""
    i = np.arange(1, n + 1)
    return 1.0 / (i[:, None] + i[None, :] - 1)
H = hilbert(8)
x_true = np.ones(8)
b = H @ x_true

In [ ]:
kappa = ...
x_solve = ...
x_inv = ...
res_solve = ...
res_inv = ...
err_solve = ...
err_inv = ...

print(f"kappa      = {kappa:.3e}")
print(f"residuals  : solve {res_solve:.3e}   inv {res_inv:.3e}")
print(f"fwd errors : solve {err_solve:.3e}   inv {err_inv:.3e}")

In [ ]:
# self-check: run this after your answer above
assert kappa > 1e9, f"kappa looks too small for an 8x8 Hilbert matrix: {kappa:.3e}"
assert x_solve.shape == (8,), f"x_solve: expected shape (8,), got {x_solve.shape}"
assert x_inv.shape == (8,), f"x_inv: expected shape (8,), got {x_inv.shape}"
_rel = lambda v: np.linalg.norm(v - x_true) / np.linalg.norm(x_true)
assert np.isclose(err_solve, _rel(x_solve), rtol=1e-6, atol=0), \
    "err_solve is not the RELATIVE forward error ||x - 1|| / ||1||—check the denominator"
assert np.isclose(err_inv, _rel(x_inv), rtol=1e-6, atol=0), \
    "err_inv is not the RELATIVE forward error ||x - 1|| / ||1||—check the denominator"
assert not np.array_equal(x_solve, x_inv), "the two routes returned bit-identical vectors—did you compute both?"
assert res_solve < 1e-12, f"res_solve should be down at floating-point noise (1e-12 or smaller), got {res_solve:.3e}"
assert res_inv > 100 * res_solve, "the inv route should have a visibly larger residual here"
assert 1e-10 < err_solve < 1e-3, f"err_solve out of the expected range: {err_solve:.3e}"
assert 1e-10 < err_inv < 1e-3, f"err_inv out of the expected range: {err_inv:.3e}"
assert err_solve > 1e6 * res_solve, "the forward error should dwarf the residual—that gap is the condition number at work"
print("looks right")

### B2. Where the accuracy goes

*Suggested: 8 minutes.*

For $n = 2, 3, \dots, 12$, build $H_n$, set $\mathbf{b} = H_n\mathbf{1}$, solve, and record two arrays of length $11$:

- `conds[i]`—the condition number of $H_n$
- `errs[i]`—the relative forward error $\|\mathbf{x}-\mathbf{1}\|_2/\|\mathbf{1}\|_2$

The cell then plots both on a log scale, with a dotted line at $10^{-6}$. Read the plot, and set `n_star` by hand to the **smallest $n$** whose error is above that line.

In [ ]:
import matplotlib.pyplot as plt
ns = np.arange(2, 13)

In [ ]:
conds = ...
errs = ...

plt.figure(figsize=(6, 3.5))
plt.semilogy(ns, errs, "o-", label="relative error")
plt.semilogy(ns, np.asarray(conds) * 1e-16, "s--", label=r"$\kappa \times 10^{-16}$")
plt.axhline(1e-6, color="gray", linestyle=":", label=r"$10^{-6}$")
plt.xticks(ns)
plt.xlabel("n")
plt.legend()
plt.tight_layout()
plt.show()

n_star = ...   # a number you read off the plot, not a formula

In [ ]:
# self-check: run this after your answer above
conds, errs = np.asarray(conds), np.asarray(errs)
assert conds.shape == (11,), f"conds: expected shape (11,), got {conds.shape}"
assert errs.shape == (11,), f"errs: expected shape (11,), got {errs.shape}"
assert np.all(np.diff(conds) > 0), "the condition numbers should increase with n—check the order of your loop"
assert conds[0] < 1e2 and conds[-1] > 1e15, "the range of condition numbers is not what a Hilbert family gives"
assert errs[0] < 1e-14, "the 2x2 case should be solved exactly, up to floating-point noise"
_expect = np.array([np.linalg.norm(np.linalg.solve(hilbert(n), hilbert(n) @ np.ones(n)) - np.ones(n))
                    / np.linalg.norm(np.ones(n)) for n in ns])
assert np.allclose(errs, _expect, rtol=1e-6, atol=0), \
    "errs is not the RELATIVE forward error ||x - 1|| / ||1|| from solve—check the denominator"
assert isinstance(n_star, (int, np.integer)), "set n_star to the whole number n you read off the plot"
i = int(n_star) - 2
assert 0 <= i < 11, f"n_star should be one of the n on the plot, 2 to 12; got {n_star}"
assert errs[i] > 1e-6, "errs at n_star does not exceed 1e-6"
assert i == 0 or errs[i - 1] <= 1e-6, "there is a smaller n that already exceeds 1e-6"
print("looks right")

### B3. Least squares through $QR$, without ever forming $A^\top A$

*Suggested: 10 minutes.*

`A` is a degree-$3$ polynomial design matrix and `y` is noisy data. Fit it by least squares **through the $QR$ decomposition**: no `lstsq`, no `pinv`, and above all no `A.T @ A`.

With $A = QR$ and $Q$ having orthonormal columns, the normal equations collapse to $R\hat\beta = Q^\top \mathbf{y}$, and $R$ is upper triangular—so this is one `solve` on a triangular system, not an inversion.

Also record `cond_A` and `cond_AtA` and compare them.

In [ ]:
rng = np.random.default_rng(6103)
xg = np.linspace(0., 4., 15)
A = np.vander(xg, 4, increasing=True)
y = 1. - 2. * xg + 0.5 * xg ** 2 + 0.3 * xg ** 3 + rng.normal(scale=0.4, size=15)

In [ ]:
Q, R = ...
beta = ...
resid = ...                                     # the residual vector y - A @ beta
cond_A = ...
cond_AtA = ...

print("beta      :", beta)
print("cond(A)   :", f"{cond_A:.4e}")
print("cond(AtA) :", f"{cond_AtA:.4e}")

In [ ]:
# self-check: run this after your answer above
assert Q.shape == (15, 4), f"Q: expected (15, 4), got {Q.shape}. The reduced QR is what you want here."
assert R.shape == (4, 4), f"R: expected (4, 4), got {R.shape}"
assert np.allclose(Q.T @ Q, np.eye(4)), "Q does not have orthonormal columns"
assert np.allclose(Q @ R, A), "Q @ R does not reproduce A"
assert np.allclose(R, np.triu(R)), "R is not upper triangular"
assert beta.shape == (4,), f"beta: expected shape (4,), got {beta.shape}"
assert resid.shape == (15,), f"resid: expected shape (15,), got {resid.shape}"
assert np.allclose(resid, y - A @ beta), "resid is not y - A @ beta"
assert np.abs(A.T @ resid).max() < 1e-9, \
    "the residual is not orthogonal to the columns of A, so beta is not the least-squares solution"
assert np.abs(Q.T @ resid).max() < 1e-9, "the residual is not orthogonal to the columns of Q either"
assert np.isclose(cond_AtA, cond_A ** 2, rtol=1e-6), \
    "cond(A.T @ A) should come out as cond(A) squared—check which matrix you conditioned"
print("looks right")

### B4. Anatomy of an SVD

*Suggested: 7 minutes.*

`X` is a random $40\times6$ matrix, except that the setup multiplies its sixth column by $10^{-14}$, so that column is all but zero. Take its **economy** SVD, then:

- reconstruct `X` from the three factors as `X_hat`
- record `rank_X` with `np.linalg.matrix_rank`
- record `cond_X` two ways: `np.linalg.cond(X)` as `cond_X`, and the ratio of singular values as `cond_from_S`

In [ ]:
X = rng.normal(size=(40, 6)) @ np.diag([10., 5., 2., 1., 0.2, 1e-14])

In [ ]:
U, S, Vt = ...
X_hat = ...
rank_X = ...
cond_X = ...
cond_from_S = ...

print("shapes:", U.shape, S.shape, Vt.shape)
print("S     :", S)
print("rank  :", rank_X, "  cond:", f"{cond_X:.3e}")

In [ ]:
# self-check: run this after your answer above
assert U.shape == (40, 6), f"U: expected (40, 6) from the economy SVD, got {U.shape}"
assert S.shape == (6,), f"S: expected (6,)—svd returns the singular values as a 1D array, got {S.shape}"
assert Vt.shape == (6, 6), f"Vt: expected (6, 6), got {Vt.shape}"
assert np.all(np.diff(S) <= 0), "the singular values should come back in descending order"
assert np.allclose(X_hat, X), "X_hat does not reconstruct X—remember S is a 1D array, not a matrix"
assert np.allclose(U.T @ U, np.eye(6)), "U should have orthonormal columns"
assert not np.allclose(U @ U.T, np.eye(40)), \
    "U @ U.T is not the identity for the economy SVD—if it is, you asked for the full one"
assert int(rank_X) == 5, f"expected numerical rank 5, got {rank_X}"
assert np.isclose(cond_X, np.linalg.cond(X), rtol=1e-6), "cond_X should be np.linalg.cond(X)"
assert np.isclose(cond_from_S, S[0] / S[-1], rtol=1e-6), \
    "cond(X) is a ratio of two particular singular values: the largest over the smallest"
assert np.isclose(cond_from_S, cond_X, rtol=1e-6), "cond(X) is a ratio of two particular singular values"
print("looks right")

### B5. Image compression by truncated SVD

*Suggested: 7 minutes.*

The setup cell makes two $240\times320$ grayscale images, each a matrix:
`img`, a picture (smooth background, flat rectangles), and `noise`, random
pixels. Compute

- `S_img` and `S_noise`: the singular values of each
- `r99` and `r99_noise`: for each image, the smallest rank $r$ whose first $r$ singular values hold $99\%$ of the total $\sum_k \sigma_k^2$

Hint: `np.cumsum`. The cell then shows both images, and each rebuilt from its first `r99` singular values.

In [ ]:
def make_image(m=240, n=320, seed=6103):
    rng = np.random.default_rng(seed)
    y, x = np.mgrid[0:m, 0:n] / n
    img = 0.55 + 0.15 * np.sin(5 * x + 2 * y)                 # smooth background
    for _ in range(45):                                       # opaque objects
        r0, c0 = rng.integers(0, m - 20), rng.integers(0, n - 20)
        h, w = rng.integers(8, 45), rng.integers(8, 45)
        img[r0:r0 + h, c0:c0 + w] = rng.uniform(0.05, 0.95)
    img += 0.02 * rng.normal(size=(m, n))                     # sensor noise
    return np.clip(img, 0., 1.)
img = make_image()
noise = np.random.default_rng(7).random(img.shape)
m, n = img.shape
def show(r):
    """img and noise, and each rebuilt from its first r singular values."""
    fig, ax = plt.subplots(2, 2, figsize=(8, 5.5))
    for row, (name, A) in enumerate([("img", img), ("noise", noise)]):
        U, S, Vt = np.linalg.svd(A, full_matrices=False)
        ax[row, 0].imshow(A, cmap="gray", vmin=0, vmax=1)
        ax[row, 0].set_title(name)
        ax[row, 1].imshow(U[:, :r] @ np.diag(S[:r]) @ Vt[:r, :], cmap="gray", vmin=0, vmax=1)
        ax[row, 1].set_title(f"{name}, first {r} singular values")
    for a in ax.flat:
        a.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
S_img = ...
S_noise = ...
r99 = ...
r99_noise = ...

print("the picture needs", r99, "of", len(S_img), "singular values")
print("the noise needs  ", r99_noise, "of", len(S_noise))
show(r99)

In [ ]:
# self-check: run this after your answer above
S_img, S_noise = np.asarray(S_img), np.asarray(S_noise)
assert np.allclose(S_img, np.linalg.svd(img, compute_uv=False)), "S_img is not the singular values of img"
assert np.allclose(S_noise, np.linalg.svd(noise, compute_uv=False)), "S_noise is not the singular values of noise"
_e = np.cumsum(S_img ** 2) / np.sum(S_img ** 2)
assert _e[int(r99) - 1] >= 0.99, \
    "the first r99 singular values of img hold less than 99% of the total; square them before adding"
assert int(r99) == 1 or _e[int(r99) - 2] < 0.99, "r99 is not the SMALLEST such rank for img"
_e = np.cumsum(S_noise ** 2) / np.sum(S_noise ** 2)
assert _e[int(r99_noise) - 1] >= 0.99, \
    "the first r99_noise singular values of noise hold less than 99% of the total"
assert int(r99_noise) == 1 or _e[int(r99_noise) - 2] < 0.99, "r99_noise is not the SMALLEST such rank for noise"
print("looks right")

## Before you leave

Upload this notebook to Blackboard. It is the record of what you did; the lab
grade itself comes from what you do in the room.

Then post a comment at the bottom of the lab page, <https://www.smajhi.com/DATS-6103/labs/numpy-II.html>. One line is enough.
It proves your GitHub account works with the comment system, which is how every
week's reading credit is earned.